# Designing a Self-Serve Platform — Runnable Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anacodicAI-labs/learn-notebooks/blob/main/system-design/11-1/self-serve-workspace-model.ipynb) &nbsp; [View on GitHub](https://github.com/anacodicAI-labs/learn-notebooks/blob/main/system-design/11-1/self-serve-workspace-model.ipynb)

Companion to: [Designing a Self-Serve Platform: One Model for Individuals and Organizations](https://anacodicai.org/learn/system-design/11-1/self-serve-workspace-model)


## What you'll build

1. One `Workspace` entity that represents **both** a personal account and an
   organization — the only difference is how many members it has.
2. One `can_access` permission function used for both, with **zero**
   conditionals on account type.
3. A direct check, on the function's own source, that it never special-cases
   by looking at `workspace.kind`.

**No API key needed.** Everything below is pure Python standard library.

### Setup

In [1]:
from dataclasses import dataclass
import inspect
print("ready — standard library only")

ready — standard library only


## Step 1 — one entity, two "kinds"

`kind` is a label only — nothing downstream is allowed to branch on it. A
personal account is a workspace with exactly one member, who owns it.

In [2]:
@dataclass
class Workspace:
    id: str
    kind: str  # "personal" or "org" -- a LABEL, never branched on
    members: dict  # user_id -> role ("owner" | "admin" | "member")

def create_personal_workspace(user_id):
    return Workspace(id=f"ws-{user_id}", kind="personal", members={user_id: "owner"})

def create_org_workspace(org_id, owner_id):
    return Workspace(id=f"ws-{org_id}", kind="org", members={owner_id: "owner"})

def invite_member(workspace, user_id, role="member"):
    workspace.members[user_id] = role

print("Workspace model ready")

Workspace model ready


## Step 2 — one permission function, used for both

**Try it:** add a new `action` (say, `"rename_workspace"`) and a rule for it —
notice you never have to touch `create_personal_workspace` or
`create_org_workspace` to do this.

In [3]:
def can_access(workspace, user_id, action):
    role = workspace.members.get(user_id)
    if role is None:
        return False
    if action == "view":
        return True
    if action == "manage_members":
        return role in ("owner", "admin")
    if action == "delete_workspace":
        return role == "owner"
    raise ValueError(f"unknown action: {action}")

print("can_access() ready")

can_access() ready


## Step 3 — a 1-member workspace and a 3-member workspace, same function

In [4]:
personal = create_personal_workspace("alice")
org = create_org_workspace("acme", owner_id="bob")
invite_member(org, "carol", role="admin")
invite_member(org, "dave", role="member")

checks = [
    (personal, "alice", "view"),
    (personal, "alice", "delete_workspace"),
    (personal, "mallory", "view"),
    (org, "bob", "delete_workspace"),
    (org, "carol", "manage_members"),
    (org, "dave", "manage_members"),
    (org, "dave", "view"),
]

print(f"{'workspace':10}{'kind':10}{'user':10}{'action':18}{'result'}")
for ws, user, action in checks:
    print(f"{ws.id:10}{ws.kind:10}{user:10}{action:18}{can_access(ws, user, action)}")

workspace kind      user      action            result
ws-alice  personal  alice     view              True
ws-alice  personal  alice     delete_workspace  True
ws-alice  personal  mallory   view              False
ws-acme   org       bob       delete_workspace  True
ws-acme   org       carol     manage_members    True
ws-acme   org       dave      manage_members    False
ws-acme   org       dave      view              True


## Step 4 — verify correctness AND verify the design property

The asserts check *behavior*. The `inspect` check below verifies the *design
claim* directly: the word `kind` never appears in `can_access`'s executable
logic — only in a comment, which we strip out first.

In [5]:
assert can_access(personal, "alice", "delete_workspace") is True
assert can_access(personal, "mallory", "view") is False
assert can_access(org, "dave", "manage_members") is False
assert can_access(org, "carol", "manage_members") is True
print("behavior asserts passed: 1-member and 3-member workspaces both correct")

source_lines = inspect.getsource(can_access).splitlines()
code_only = "\n".join(line.split("#")[0] for line in source_lines)
assert "kind" not in code_only
print("design assert passed: can_access() never references workspace.kind "
      "in its actual logic")

behavior asserts passed: 1-member and 3-member workspaces both correct
design assert passed: can_access() never references workspace.kind in its actual logic


## Try it yourself

- Add a fourth workspace with zero members added yet (just the owner) and
  confirm every check still works with no code changes.
- Try adding an org-only feature (say, `"configure_sso"`) that should only be
  allowed when `len(workspace.members) > 1` — notice this is a *data*
  condition on the workspace, not a fork into a separate account-type system.